# CardioIA — Parte 1: Modelo preditivo de `pico_risco`

Base sintética, treino supervisionado, métricas e exportação do modelo para integração multiagente.

**Executar no Google Colab:** `Arquivo > Fazer upload do notebook` ou copiar células; instalar dependências na primeira célula de código.

In [ ]:
# Instalação (Colab)
%pip install -q numpy pandas scikit-learn joblib matplotlib seaborn

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)
RANDOM_STATE = 42

## 1. Geração da base sintética

Variáveis alinhadas ao enunciado: idade, frequência cardíaca, SpO2, carga do sistema, disponibilidade de recursos; alvo binário `pico_risco`.

In [ ]:
def gerar_base_sintetica(n=2000, seed=42):
    rng = np.random.default_rng(seed)
    idade = rng.integers(35, 90, size=n)
    freq_cardiaca = rng.normal(78, 18, size=n).clip(45, 180)
    spo2 = rng.normal(96, 3, size=n).clip(70, 100)
    carga_sistema = rng.uniform(0, 100, size=n)
    disponibilidade_recursos = rng.uniform(0, 100, size=n)

    score_latente = (
        0.02 * (idade - 40)
        + 0.015 * np.maximum(0, freq_cardiaca - 75)
        - 0.08 * (spo2 - 92)
        + 0.012 * carga_sistema
        - 0.01 * disponibilidade_recursos
        + rng.normal(0, 0.35, size=n)
    )
    prob = 1 / (1 + np.exp(-score_latente))
    pico_risco = (rng.random(n) < prob).astype(int)

    df = pd.DataFrame(
        {
            "idade": idade.astype(float),
            "freq_cardiaca": freq_cardiaca,
            "spo2": spo2,
            "carga_sistema": carga_sistema,
            "disponibilidade_recursos": disponibilidade_recursos,
            "pico_risco": pico_risco,
        }
    )
    return df


df = gerar_base_sintetica(2500, RANDOM_STATE)
df.head(10)

In [ ]:
df["pico_risco"].value_counts(normalize=True)

## 2. Preparação e separação treino / teste

In [ ]:
FEATURES = [
    "idade",
    "freq_cardiaca",
    "spo2",
    "carga_sistema",
    "disponibilidade_recursos",
]
TARGET = "pico_risco"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

## 3. Treinamento — Random Forest (classificação supervisionada)

In [ ]:
model = RandomForestClassifier(
    n_estimators=120,
    max_depth=12,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)
model.fit(X_train, y_train)

## 4. Avaliação (acurácia, matriz de confusão, relatório)

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print(f"Acurácia: {acc:.4f}")
print("Matriz de confusão:\n", cm)
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0, 1], yticklabels=[0, 1])
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de confusão — pico_risco")
plt.tight_layout()
plt.show()

## 5. Persistência do modelo + metadados (integração multiagente)

In [ ]:
artifact = {
    "model": model,
    "features": FEATURES,
    "target": TARGET,
    "metrics": {"accuracy_test": float(acc), "confusion_matrix": cm.tolist()},
}

out_dir = Path("/content") if Path("/content").exists() else Path(".")
out_path = out_dir / "cardio_pico_risco_artifact.joblib"
joblib.dump(artifact, out_path)
print("Salvo em:", out_path.resolve())

## 6. Simulação — novo paciente (interpretação da probabilidade)

In [ ]:
novo_paciente = pd.DataFrame(
    [
        {
            "idade": 72.0,
            "freq_cardiaca": 118.0,
            "spo2": 88.0,
            "carga_sistema": 82.0,
            "disponibilidade_recursos": 35.0,
        }
    ]
)

p_pico = float(model.predict_proba(novo_paciente[FEATURES])[:, 1])
classe = int(model.predict(novo_paciente[FEATURES])[0])
print("Paciente simulado:")
print(novo_paciente.T)
print(f"\nProbabilidade estimada de pico_risco=1: {p_pico:.3f}")
print(f"Classe predita (0/1): {classe}")